In [ ]:
import sys
sys.path.append("../")
import numpy as np
from utils.noise_generator import ColoredNoiseGenerator_Cholesky

def build_C_from_alpha(alpha, tgrid):
    N = len(tgrid)
    C = np.zeros((N, N), dtype=complex)
    for i in range(N):
        for j in range(N):
            C[i, j] = alpha(tgrid[j] - tgrid[i])
    return C

def rel_fro_error(A, B, eps=1e-15):
    return np.linalg.norm(A - B, 'fro') / (np.linalg.norm(B, 'fro') + eps)

def alpha(t):
    g = 2
    w = 0.5 + 2j
    a = lambda t: g * np.exp(-w * t)
    if t >= 0:
        return a(t)
    else:
        return np.conj(a(-t))

steps = 10
tmax = np.pi
dt = tmax / steps
tgrid = np.linspace(0, tmax, steps + 1)

C = build_C_from_alpha(alpha, tgrid)

noise_generator = ColoredNoiseGenerator_Cholesky(alpha, t_stop=tmax, N_steps=steps)

N_samples = 100000
C_hat = 0

for n in range(N_samples):
    z = noise_generator.sample_process()
    C_hat += np.outer(z, z.conj())

C_hat /= N_samples

print("Relative Frobenius error:", rel_fro_error(C_hat, C))

In [ ]:
# bath correlation function in Cai_2020_CPAM
Delta = 1
beta = 5/Delta
wc = 2.5 * Delta
wmax = 4 * wc
CapL = 200
factor = 1 - np.exp(-wmax/wc)
wl = -wc * np.log(1 - np.linspace(1, CapL, CapL)/CapL * factor)
cl = wl * np.sqrt((0.2 * wc/CapL) * factor)

_coth = 1.0 / np.tanh(0.5 * beta * wl)
_pref = (cl**2) / (2.0 * wl)
def bath_corr(t):
    """
    Bath correlation function alpha(t).
    Supports scalar t or 1D array t (returns same shape).
    """
    # t_arr = np.asarray(t)
    # t1 = np.atleast_1d(t_arr).astype(float)
    # phase = wl[:, None] * t1[None, :]             # shape: (CapL, len(t))
    # out = np.sum(_pref[:, None] * (np.cos(phase) * _coth[:, None] - 1j * np.sin(phase)), axis=0)
    # return out[0] if t_arr.ndim == 0 else out
    res = 0
    for l in range(CapL):
        res += _pref[l] * (_coth[l] * np.cos(wl[l]*t) - 1j * np.sin(wl[l]*t))
    return res

tmax = 5/Delta
steps = 50
dt = tmax / steps
tgrid = np.linspace(0, tmax, steps + 1)


noise_generator = ColoredNoiseGenerator_Cholesky(bath_corr, t_stop=tmax, N_steps=steps)

In [ ]:
base = 50
rank = 8
N_steps = np.array([1, 2, 4,]) * base

for steps in N_steps:
    tgrid = np.linspace(0, tmax, steps + 1)
    C = np.zeros((steps + 1, steps + 1), dtype=complex)
    for i in range(steps + 1):
        for j in range(steps + 1):
            C[i, j] = bath_corr(tgrid[i] - tgrid[j])
    U, S, V = np.linalg.svd(C)
    r = rank * (steps // base)
    print(f"Steps: {steps}, first few singular values: {S[:2*(steps//base)]}")
    # print(f"first few vectors: {U[:2*(steps//base), :2*(steps//base)]}")
    C_approx = U[:, :r] @ np.diag(S[:r]) @ V[:r, :]
    print(f"Steps: {steps}, Relative Frobenius error of rank-{r} approximation: {rel_fro_error(C_approx, C)}")
    C_approx = U[:, :rank] @ np.diag(S[:rank]) @ V[:rank, :]
    print(f"Steps: {steps}, Relative Frobenius error of rank-{rank} approximation: {rel_fro_error(C_approx, C)}")

In [ ]:
base = 50
N_steps = np.array([1, 2, 4,]) * base
rank = 8

C = np.zeros((N_steps[-1] + 1, N_steps[-1] + 1), dtype=complex)
tgrid = np.linspace(0, tmax, N_steps[-1] + 1)
for i in range(N_steps[-1] + 1):
    for j in range(N_steps[-1] + 1):
        C[i, j] = bath_corr(tgrid[i] - tgrid[j])
U, S, V = np.linalg.svd(C)
U_r = U[:, :rank]
V_r = V[:rank, :]
S_r = S[:rank]

for steps in N_steps[:-1]:
    tgrid = np.linspace(0, tmax, steps + 1)
    C_small = np.zeros((steps + 1, steps + 1), dtype=complex)
    for i in range(steps + 1):
        for j in range(steps + 1):
            C_small[i, j] = bath_corr(tgrid[i] - tgrid[j])

    U_, S_, V_ = np.linalg.svd(C_small)
    C_small_svd = U_[:, :rank] @ np.diag(S_[:rank]) @ V_[:rank, :]

    ds = N_steps[-1] // steps
    C_small_app = U_r[::ds, :] @ np.diag(S_r) @ V_r[:, ::ds]
    print(f"Steps: {steps}, Relative Frobenius error of rank-{rank} approximation: {rel_fro_error(C_small_app, C_small)}")
    print(f"Steps: {steps}, Relative Frobenius error of rank-{rank} approximation (SVD vs interp): {rel_fro_error(C_small_app, C_small_svd)}")


In [ ]:
import matplotlib.pyplot as plt

# 更详细的奇异值分析
base = 50
N_steps_array = np.array([1, 2, 4]) * base
rank = 4

singular_values_dict = {}

for steps in N_steps_array:
    tgrid = np.linspace(0, tmax, steps + 1)
    dt = tmax / steps
    
    C = np.zeros((steps + 1, steps + 1), dtype=complex)
    for i in range(steps + 1):
        for j in range(steps + 1):
            C[i, j] = bath_corr(tgrid[i] - tgrid[j])
    
    U, S, V = np.linalg.svd(C)
    singular_values_dict[steps] = S
    
    print(f"\n步数: {steps}, dt: {dt:.4f}")
    print(f"  矩阵大小: ({steps+1}, {steps+1})")
    print(f"  前4个奇异值: {S[:4]}")
    if steps // 2 in singular_values_dict:
        ratio = S[0] / singular_values_dict[steps // 2][0]
        print(f"  奇异值与前一个步数的比率: {ratio:.4f}x")
    print(f"  Frobenius范数: {np.linalg.norm(S):.4f}")

# 绘制奇异值对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：绝对奇异值
for steps in N_steps_array:
    S = singular_values_dict[steps]
    axes[0].semilogy(S[:min(20, len(S))], marker='o', label=f'steps={steps}, dt={tmax/steps:.4f}')

axes[0].set_xlabel('Index')
axes[0].set_ylabel('Singular Values (log scale)')
axes[0].set_title('Singular Values vs Index')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 右图：归一化后的奇异值（除以步数）
for steps in N_steps_array:
    S = singular_values_dict[steps]
    # 尝试除以不同的因子
    S_normalized = S / steps  # 按步数归一化
    axes[1].semilogy(S_normalized[:min(20, len(S_normalized))], marker='s', label=f'steps={steps}')

axes[1].set_xlabel('Index')
axes[1].set_ylabel('Normalized Singular Values (S/steps)')
axes[1].set_title('Normalized Singular Values (Should Collapse if S ∝ steps)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 分析缩放关系
print("\n" + "="*60)
print("缩放关系分析:")
print("="*60)
steps_ref = N_steps_array[0]
S_ref = singular_values_dict[steps_ref]

for i in range(min(4, len(S_ref))):
    print(f"\n第{i+1}个奇异值:")
    for steps in N_steps_array:
        S = singular_values_dict[steps]
        ratio = steps / steps_ref
        S_scaled = S[i]
        S_expected = S_ref[i] * ratio
        print(f"  steps={steps:3d}: σ={S_scaled:8.4f}, 期望值(若σ∝steps)={S_expected:8.4f}, 比率={S_scaled/S_expected:.4f}")


## 奇异值缩放现象的物理与数学解释

### 观察到的现象
- **奇异值与步数的线性关系**：$\sigma_k \propto \text{steps}$ 或等价地 $\sigma_k \propto 1/dt$
- **缩放比例**：当步数翻倍时，奇异值也翻倍（比率在0.99以上）
- **Frobenius范数**：$\|C\|_F = \sqrt{\sum_i \sigma_i^2}$ 也随步数线性增长

### 数学原因

相关矩阵定义为：
$$C_{ij} = \alpha(t_i - t_j)$$

其中 $t_i = i \cdot dt$，$i = 0, 1, ..., N$。

当 $dt$ 减小时：
1. **矩阵大小增加**：从 $(N+1) \times (N+1)$ 变为 $(2N+1) \times (2N+1)$
2. **样本密度增加**：在相同的时间区间内，采样点数增加
3. **离散和相关**：$\|C\|_F^2 = \sum_{i,j} |\alpha(t_i - t_j)|^2 \approx \frac{1}{dt^2} \int_0^{t_{max}} \int_0^{t_{max}} |\alpha(s-r)|^2 \, ds \, dr$

因此 $\|C\|_F \propto 1/dt$，这导致奇异值也 $\propto 1/dt$。

### 物理意义

这个现象表明了**离散化对"能量"或"相关强度"的影响**：

1. **记忆效应的分辨率**：
   - 当 dt 更小时，我们以更高的时间分辨率捕捉系统的记忆效应
   - 每个时间点上相关函数的"累积"增加

2. **低秩近似的有效性**：
   - 矩阵条件数 $\kappa = \sigma_{max}/\sigma_{min}$ 保持相对稳定（归一化后的奇异值基本不变）
   - 这意味着我们可以用相同的低秩来近似不同分辨率的问题
   - 例如：rank-4 近似对 steps=50，100，200 都可能是有效的

3. **数值积分和离散化**：
   - 这反映了黎曼和对积分的近似
   - dt 越小，离散近似越精确，但"累积"的相关值也越大

4. **对算法的影响**：
   - 如果使用了低秩分解（如 `NMLRSSE`），秩的选择应该**不依赖于 dt**
   - 相反，应该基于**物理系统的特征时间尺度**，而非数值离散化参数


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def em_gbm_paths(mu, sigma, x0, T, N, M, rng):
    dt = T / N
    dW = rng.normal(0.0, np.sqrt(dt), size=(M, N))
    W_T = dW.sum(axis=1)

    X = np.full(M, x0, dtype=np.float64)
    for n in range(N):
        X = X + mu * X * dt + sigma * X * dW[:, n]

    return X, W_T

def exact_gbm_terminal(mu, sigma, x0, T, W_T):
    return x0 * np.exp((mu - 0.5 * sigma**2) * T + sigma * W_T)

def fit_order(dts, errs):
    x = np.log(dts)
    y = np.log(errs)
    p, _ = np.polyfit(x, y, 1)
    return p

def main():
    mu = 2.0
    sigma = 1.0
    x0 = 1.0
    T = 1.0

    M = 200000
    rng = np.random.default_rng(12345)

    ks = np.array([4,5,6,7,8,9,10])
    Ns = 2**ks
    dts = T / Ns

    strong_err = []
    weak_err = []

    exact_EX = x0 * np.exp(mu * T)

    for N in Ns:
        X_em_T, W_T = em_gbm_paths(mu, sigma, x0, T, N, M, rng)
        X_ex_T = exact_gbm_terminal(mu, sigma, x0, T, W_T)

        strong_err.append(np.mean(np.abs(X_em_T - X_ex_T)))
        weak_err.append(np.abs(np.mean(X_em_T) - exact_EX))

    strong_err = np.array(strong_err)
    weak_err = np.array(weak_err)

    p_strong = fit_order(dts, strong_err)
    p_weak = fit_order(dts, weak_err)

    print(f"Observed strong order ≈ {p_strong:.3f}")
    print(f"Observed weak   order ≈ {p_weak:.3f}")

    # ===== 单张图 =====
    plt.figure(figsize=(8,6))

    plt.loglog(dts, strong_err, 'o-', label=f'Strong error (≈{p_strong:.2f})')
    plt.loglog(dts, weak_err, 's-', label=f'Weak error (≈{p_weak:.2f})')

    # 参考线
    C1 = strong_err[0] / (dts[0]**0.5)
    plt.loglog(dts, C1 * dts**0.5, '--', label='Ref slope 1/2')

    C2 = weak_err[0] / (dts[0]**1.0)
    plt.loglog(dts, C2 * dts**1.0, '--', label='Ref slope 1')

    plt.gca().invert_xaxis()
    plt.xlabel('dt')
    plt.ylabel('Error')
    plt.title('Euler–Maruyama Convergence Test')
    plt.legend()
    plt.grid(True, which='both', linestyle=':')

    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Proper complex Gaussian colored noise z(t)
# Build x(t), y(t) as independent real Gaussian processes with same covariance alpha.
# Then z(t) = (x(t) + i y(t))/sqrt(2) is proper:
#   E[z]=0, E[z z]=0, E[z(t) z(s)^*] = alpha(t,s)
# -----------------------------

def sample_real_process_coeffs(M, K, rng, S_k):
    """
    Random Fourier series coefficients for a real stationary Gaussian process:
      x(t) = sum_k sqrt(2 S_k) [a_k cos(w_k t) + b_k sin(w_k t)]
    with a_k, b_k iid N(0,1). This yields a band-limited smooth process.
    """
    a = rng.normal(size=(M, K))
    b = rng.normal(size=(M, K))
    scale = np.sqrt(2.0 * S_k)[None, :]  # (1,K)
    return scale * a, scale * b

def precompute_real_on_grid(tgrid, a, b, w):
    """
    Precompute x(t_n) for all paths on a time grid.
    a,b: (M,K), w: (K,), tgrid: (N,)
    return X: (M,N)
    """
    cosmtx = np.cos(np.outer(tgrid, w))  # (N,K)
    sinmtx = np.sin(np.outer(tgrid, w))  # (N,K)
    # (M,K) @ (K,N) -> (M,N)
    return a @ cosmtx.T + b @ sinmtx.T

def make_proper_complex_noise_on_grid(tgrid, coeffs, w):
    """
    coeffs = (a_x,b_x,a_y,b_y) with x,y independent and same spectrum.
    returns Z: (M,N) complex, Z(t) = (x(t)+i y(t))/sqrt(2)
    """
    a_x, b_x, a_y, b_y = coeffs
    X = precompute_real_on_grid(tgrid, a_x, b_x, w)
    Y = precompute_real_on_grid(tgrid, a_y, b_y, w)
    return (X + 1j * Y) / np.sqrt(2.0)

# -----------------------------
# Random ODE: X' = mu X + sigma X z(t)
# Euler scheme on grid: X_{n+1} = X_n + (mu X_n + sigma X_n z(t_n)) dt
# Reference: X(T) = x0 * exp(mu T + sigma * integral z dt)
# -----------------------------

def euler_random_ode(mu, sigma, x0, T, Zgrid):
    """
    Zgrid: (M, N+1) complex values at t_n = n*dt
    Euler uses z(t_n) for n=0..N-1
    returns X_T (M,)
    """
    M, Np1 = Zgrid.shape
    N = Np1 - 1
    dt = T / N
    X = np.full(M, x0, dtype=np.complex128)
    for n in range(N):
        zt = Zgrid[:, n]
        X = X + (mu * X + sigma * X * zt) * dt
    return X

def reference_solution(mu, sigma, x0, T, Zref):
    """
    Zref: (M, Nref+1) complex values on fine grid
    Uses trapezoidal rule for integral_0^T z(t) dt.
    returns X_ref(T) (M,)
    """
    M, Np1 = Zref.shape
    Nref = Np1 - 1
    dt = T / Nref
    I = dt * (0.5 * Zref[:, 0] + Zref[:, 1:-1].sum(axis=1) + 0.5 * Zref[:, -1])
    return x0 * np.exp(mu * T + sigma * I)

def fit_order(dts, errs):
    x = np.log(dts)
    y = np.log(errs)
    p, _ = np.polyfit(x, y, 1)
    return p

def main():
    # ---- ODE params ----
    mu = 1.0
    sigma = 0.8
    x0 = 1.0
    T = 1.0

    # ---- Monte Carlo ----
    M = 50000           # increase if weak tail is noisy
    rng = np.random.default_rng(2026)

    # ---- Noise spectrum (band-limited => smooth) ----
    K = 60
    w0 = 2.0 * np.pi / T
    w = w0 * np.arange(1, K + 1)  # w_k

    # Choose a decaying spectrum S_k to define alpha(t,s)=alpha(t-s) implicitly.
    # Faster decay => smoother. This induces a stationary covariance kernel.
    k = np.arange(1, K + 1)
    S_k = 1.0 / (1.0 + k**4)

    # Sample independent coefficients for x(t), y(t) with same spectrum
    a_x, b_x = sample_real_process_coeffs(M, K, rng, S_k)
    a_y, b_y = sample_real_process_coeffs(M, K, rng, S_k)
    coeffs = (a_x, b_x, a_y, b_y)

    # ---- Reference grid ----
    Nref = 2**16
    t_ref = np.linspace(0.0, T, Nref + 1)
    Z_ref = make_proper_complex_noise_on_grid(t_ref, coeffs, w)
    X_ref = reference_solution(mu, sigma, x0, T, Z_ref)

    # ---- Test grids ----
    ks = np.array([4, 5, 6, 7, 8, 9, 10], dtype=int)  # N=16..1024
    Ns = 2**ks
    dts = T / Ns

    strong_err = []
    weak_err = []

    for N in Ns:
        t = np.linspace(0.0, T, N + 1)
        Z = make_proper_complex_noise_on_grid(t, coeffs, w)
        X_num = euler_random_ode(mu, sigma, x0, T, Z)

        # strong: E |X_num - X_ref|
        se = np.mean(np.abs(X_num - X_ref))
        # weak: |E[X_num] - E[X_ref]|
        we = np.abs(np.mean(X_num) - np.mean(X_ref))

        strong_err.append(se)
        weak_err.append(we)
        print(f"N={N:4d}, dt={T/N:.3e}, strong={se:.3e}, weak={we:.3e}")

    strong_err = np.array(strong_err)
    weak_err = np.array(weak_err)

    pS = fit_order(dts, strong_err)
    pW = fit_order(dts, weak_err)
    print(f"\nObserved orders: strong ≈ {pS:.3f}, weak ≈ {pW:.3f}")

    # ---- Plot both on one figure + reference slope 1 ----
    plt.figure(figsize=(8, 6))
    plt.loglog(dts, strong_err, "o-", label=f"Strong error (≈{pS:.2f})")
    plt.loglog(dts, weak_err,   "s-", label=f"Weak error   (≈{pW:.2f})")

    # Reference slope 1 line (anchored to first strong point)
    C = strong_err[0] / dts[0]
    plt.loglog(dts, C * dts, "--", label="Ref slope 1")

    plt.gca().invert_xaxis()
    plt.xlabel("dt")
    plt.ylabel("Error")
    plt.title("Euler convergence with proper complex colored noise: strong & weak ~ 1")
    plt.grid(True, which="both", linestyle=":")
    plt.legend()
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()